In [2]:
import pandas as pd
import numpy as np

def load_data(data_path='data/raw/'):
    """加载原始数据"""
    item_feature = pd.read_csv(f'{data_path}Item_feature.csv')
    cost_data = pd.read_csv(f'{data_path}Cost.csv')
    meta_data = pd.read_csv(f'{data_path}bwds_meta.csv')
    return item_feature, cost_data, meta_data

def enhanced_preprocess_data(item_feature, meta_data):
    """增强的数据预处理函数"""
    item_feature['date'] = pd.to_datetime(item_feature['date'])
    
    if 'item_id' in meta_data.columns:
        item_feature = item_feature.merge(meta_data[['item_id']], on='item_id', how='left')
    
    numeric_cols = item_feature.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if item_feature[col].isnull().sum() > 0:
            item_feature[col] = item_feature[col].fillna(item_feature[col].median())
    
    item_feature['region_id'] = (item_feature['item_id'] % 5 + 1).astype(int)
    
    # 时间特征
    item_feature['year'] = item_feature['date'].dt.year
    item_feature['month'] = item_feature['date'].dt.month
    item_feature['day'] = item_feature['date'].dt.day
    item_feature['dayofweek'] = item_feature['date'].dt.dayofweek
    item_feature['weekend'] = (item_feature['date'].dt.dayofweek >= 5).astype(int)
    item_feature['quarter'] = item_feature['date'].dt.quarter
    item_feature['is_month_start'] = item_feature['date'].dt.is_month_start.astype(int)
    item_feature['is_month_end'] = item_feature['date'].dt.is_month_end.astype(int)
    item_feature['dayofyear'] = item_feature['date'].dt.dayofyear
    
    # 季节性特征
    item_feature['month_sin'] = np.sin(2 * np.pi * item_feature['month']/12)
    item_feature['month_cos'] = np.cos(2 * np.pi * item_feature['month']/12)
    item_feature['dayofyear_sin'] = np.sin(2 * np.pi * item_feature['dayofyear']/365)
    item_feature['dayofyear_cos'] = np.cos(2 * np.pi * item_feature['dayofyear']/365)
    
    return item_feature

def create_enhanced_features(df):
    """创建增强的特征工程"""
    df = df.copy()
    
    # 用户行为比率特征
    df['cart_conversion_rate'] = df['num_alipay'] / (df['cart_ipv'] + 1)
    df['pv_conversion_rate'] = df['num_alipay'] / (df['pv_uv'] + 1)
    df['collect_conversion_rate'] = df['num_alipay'] / (df['collect_uv'] + 1)
    
    # 营销效果特征
    df['ztc_effectiveness'] = df['ztc_pv_ipv'] / (df['pv_uv'] + 1)
    df['tbk_effectiveness'] = df['tbk_pv_ipv'] / (df['pv_uv'] + 1)
    df['ss_effectiveness'] = df['ss_pv_ipv'] / (df['pv_uv'] + 1)
    df['jhs_effectiveness'] = df['jhs_pv_ipv'] / (df['pv_uv'] + 1)
    
    # 价格特征
    df['price_indicator'] = df['amt_alipay'] / (df['qty_alipay'] + 1)
    
    # 历史统计特征
    df['item_avg_sales'] = df.groupby('item_id')['qty_alipay'].transform('mean')
    df['cate_avg_sales'] = df.groupby('cate_level_id')['qty_alipay'].transform('mean')
    df['region_avg_sales'] = df.groupby('region_id')['qty_alipay'].transform('mean')
    
    # 滞后与移动平均特征
    df_sorted = df.sort_values(['item_id', 'date'])
    for lag in [1, 3, 7]:
        df_sorted[f'lag_{lag}'] = df_sorted.groupby('item_id')['qty_alipay'].shift(lag)
    for window in [3, 7, 14]:
        df_sorted[f'rolling_mean_{window}'] = df_sorted.groupby('item_id')['qty_alipay'].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean()
        )
    
    return df_sorted

def prepare_train_test_data(model_data, test_start_date='2015-12-14', test_end_date='2015-12-27'):
    """划分时间序列训练集和测试集"""
    model_data = model_data.sort_values('date')
    train_data = model_data[model_data['date'] < pd.to_datetime(test_start_date)]
    test_data = model_data[(model_data['date'] >= pd.to_datetime(test_start_date)) & 
                           (model_data['date'] <= pd.to_datetime(test_end_date))]
    return train_data, test_data